# Bayesian Neural Networks: Laplace and the Evidence Framework

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/bayesian_neural_networks.ipynb)

Companion notebook to the blog post [Bayesian Neural Networks: Laplace and Evidence Framework](https://sesen.ai/blog/bayesian-neural-networks-laplace-evidence).

We turn a two-layer GELU network into a Bayesian neural network using Bishop PRML §5.7:
- Train to $\mathbf{w}_{\text{MAP}}$ with a Gaussian prior (weight decay).
- Compute the Hessian $\mathbf{H}$ of the sum-of-squared-errors at $\mathbf{w}_{\text{MAP}}$ via `torch.autograd.functional.hessian`.
- Form the posterior precision $\mathbf{A} = \alpha \mathbf{I} + \beta \mathbf{H}$, approximate $p(\mathbf{w}|\mathcal{D}) \approx \mathcal{N}(\mathbf{w}_{\text{MAP}}, \mathbf{A}^{-1})$ (Laplace approximation).
- Linearise the predictive to get closed-form $\pm 2\sigma$ bands: $\sigma^2(\mathbf{x}) = \beta^{-1} + \mathbf{g}^\top \mathbf{A}^{-1} \mathbf{g}$.
- Close the loop with MacKay's evidence framework to choose $\alpha$ and $\beta$ from the data.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.autograd.functional import hessian
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

## 1. The toy dataset

25 noisy observations of $\sin(2\pi x)$ on two disjoint intervals. The gap in the middle and the empty regions at the tails give the BNN room to show inflated uncertainty where no data constrains it.

In [ ]:
SIGMA_TRUE = 0.12
x_train_np = np.concatenate([
    np.linspace(-0.9, -0.35, 12),
    np.linspace(0.1, 0.7, 13),
])
y_train_np = np.sin(2 * np.pi * x_train_np) + SIGMA_TRUE * np.random.randn(len(x_train_np))

x = torch.tensor(x_train_np).unsqueeze(1)
y = torch.tensor(y_train_np).unsqueeze(1)

xs_np = np.linspace(-1.0, 1.0, 400)
plt.figure(figsize=(7, 3.5))
plt.plot(xs_np, np.sin(2 * np.pi * xs_np), 'k--', alpha=0.5, label='true sin(2πx)')
plt.scatter(x_train_np, y_train_np, color='#D4A24C', edgecolor='#1B2D3D', s=40, zorder=5, label='observations')
plt.xlabel('x'); plt.ylabel('y'); plt.grid(alpha=0.3); plt.legend()
plt.show()

## 2. The network

Two-layer GELU MLP with 12 hidden units (37 parameters total). We pack all parameters into a single flat tensor `theta` so that `torch.autograd.functional.hessian` can differentiate a scalar loss with respect to a single vector argument.

In [ ]:
HIDDEN = 12
W = HIDDEN + HIDDEN + HIDDEN + 1  # 37 parameters

def predict(theta, x_in):
    W1 = theta[:HIDDEN].view(HIDDEN, 1)
    b1 = theta[HIDDEN:2*HIDDEN]
    W2 = theta[2*HIDDEN:3*HIDDEN].view(1, HIDDEN)
    b2 = theta[3*HIDDEN:]
    return F.gelu(x_in @ W1.T + b1) @ W2.T + b2

def sse(theta):
    return 0.5 * ((predict(theta, x) - y) ** 2).sum()

def neg_log_posterior(theta, alpha, beta):
    return beta * sse(theta) + 0.5 * alpha * (theta @ theta)

## 3. Training loop

Adam for warm-up, L-BFGS for a tight final convergence. This is the recipe Bishop §5.7 calls for: we need a good local mode so that the Laplace approximation is meaningful.

In [ ]:
def train(loss_fn, steps=6000, lr=0.02, seed=1):
    torch.manual_seed(seed)
    theta = (0.3 * torch.randn(W)).detach().clone().requires_grad_(True)
    opt = torch.optim.Adam([theta], lr=lr)
    for _ in range(steps):
        opt.zero_grad(); l = loss_fn(theta); l.backward(); opt.step()
    opt2 = torch.optim.LBFGS([theta], lr=1.0, max_iter=200, tolerance_grad=1e-10)
    def closure():
        opt2.zero_grad(); l = loss_fn(theta); l.backward(); return l
    opt2.step(closure)
    return theta.detach().clone()

BETA_INIT = 1.0 / SIGMA_TRUE ** 2
ALPHA_INIT = 2.0

theta_mle = train(lambda t: BETA_INIT * sse(t))
theta_map = train(lambda t: neg_log_posterior(t, ALPHA_INIT, BETA_INIT))
print('MLE SSE:', sse(theta_mle).item(), '  MAP SSE:', sse(theta_map).item())

## 4. Laplace posterior at the MAP

Compute $\mathbf{H} = \nabla\nabla \, \text{SSE}(\mathbf{w})|_{\mathbf{w}_{\text{MAP}}}$ using PyTorch autograd. Then $\mathbf{A} = \alpha\mathbf{I} + \beta\mathbf{H}$. For the first pass we use the starting hyperparameters; later we refine them with the evidence framework.

In [ ]:
H = hessian(sse, theta_map).detach().numpy()
H = 0.5 * (H + H.T)
print('H shape:', H.shape, '  symmetric:', np.allclose(H, H.T))

def posterior_covariance(H, alpha, beta):
    A = alpha * np.eye(H.shape[0]) + beta * H
    return np.linalg.inv(0.5 * (A + A.T))

## 5. Predictive band

For each test input $\mathbf{x}$ we need the gradient $\mathbf{g}(\mathbf{x}) = \nabla_\mathbf{w} y(\mathbf{x}, \mathbf{w})|_{\mathbf{w}_{\text{MAP}}}$. Each gradient is one backward pass through the network.

In [ ]:
def predict_np(theta_np, x_eval):
    t = torch.tensor(theta_np)
    return predict(t, torch.tensor(x_eval).unsqueeze(1)).detach().numpy().ravel()

def jacobian_wrt_theta(theta_tensor, x_scalar):
    xs = torch.tensor([[float(x_scalar)]])
    t = theta_tensor.detach().clone().requires_grad_(True)
    y_out = predict(t, xs).squeeze()
    (g,) = torch.autograd.grad(y_out, t)
    return g.detach().numpy()

def predictive_band(theta_map_t, H, alpha, beta, x_eval):
    A_inv = posterior_covariance(H, alpha, beta)
    mean = predict_np(theta_map_t.numpy(), x_eval)
    var = np.empty_like(mean)
    for i, xi in enumerate(x_eval):
        g = jacobian_wrt_theta(theta_map_t, xi)
        var[i] = 1.0 / beta + g @ A_inv @ g
    return mean, np.sqrt(np.clip(var, 0, None))

## 6. MLE vs MAP vs Bayesian (three-panel figure)

In [ ]:
mean_band, std_band = predictive_band(theta_map, H, ALPHA_INIT, BETA_INIT, xs_np)
y_mle = predict_np(theta_mle.numpy(), xs_np)
y_map = predict_np(theta_map.numpy(), xs_np)
y_true = np.sin(2 * np.pi * xs_np)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, (lbl, curve) in zip(axes, [
    ('MLE (no prior)', y_mle),
    ('MAP (ridge)', y_map),
    ('Bayesian (Laplace)', mean_band),
]):
    ax.plot(xs_np, y_true, 'k--', alpha=0.5, label='true')
    ax.scatter(x_train_np, y_train_np, color='#D4A24C', edgecolor='#1B2D3D', s=38, zorder=5)
    ax.plot(xs_np, curve, color='#B44', lw=2.2, label='prediction')
    if lbl == 'Bayesian (Laplace)':
        ax.fill_between(xs_np, mean_band - 2*std_band, mean_band + 2*std_band, color='#B44', alpha=0.18, label='±2σ')
    ax.set_title(lbl); ax.set_xlabel('x'); ax.grid(alpha=0.3); ax.set_ylim(-2.2, 2.2)
    ax.legend(fontsize=8)
axes[0].set_ylabel('y')
plt.tight_layout(); plt.show()

## 7. Evidence framework

Iterate the MacKay (1992) re-estimation formulas:

$$\alpha_{\text{new}} = \frac{\gamma}{\mathbf{w}^\top \mathbf{w}}, \quad \frac{1}{\beta_{\text{new}}} = \frac{1}{N - \gamma} \sum_n \{y_n - t_n\}^2, \quad \gamma = \sum_i \frac{\lambda_i}{\alpha + \lambda_i}.$$

After each α,β update we retrain to $\mathbf{w}_{\text{MAP}}$ and recompute the Hessian.

In [ ]:
alpha, beta = 1e-3, 1.0 / SIGMA_TRUE ** 2
N = len(x_train_np)
log = []
for step in range(10):
    theta_cur = train(lambda t: neg_log_posterior(t, alpha, beta), steps=2000)
    H_cur = hessian(sse, theta_cur).detach().numpy()
    H_cur = 0.5 * (H_cur + H_cur.T)
    eig_bH = np.clip(np.linalg.eigvalsh(beta * H_cur), 0, None)
    gamma = np.sum(eig_bH / (alpha + eig_bH))
    w_np = theta_cur.numpy()
    yhat = predict_np(w_np, x_train_np)
    sse_val = ((yhat - y_train_np) ** 2).sum()
    alpha_new = gamma / (w_np @ w_np + 1e-12)
    beta_new = max(N - gamma, 1.0) / sse_val
    # log marginal likelihood (Bishop 5.175)
    A_mat = alpha * np.eye(W) + beta * H_cur
    _, logdetA = np.linalg.slogdet(A_mat)
    E = 0.5 * beta * sse_val + 0.5 * alpha * (w_np @ w_np)
    log_ev = -E - 0.5 * logdetA + 0.5 * W * np.log(alpha) + 0.5 * N * np.log(beta) - 0.5 * N * np.log(2*np.pi)
    log.append(dict(step=step, alpha=alpha, beta=beta, gamma=gamma, log_ev=log_ev))
    print(f'step {step:2d}:  α={alpha:.4f}  β={beta:.3f}  γ={gamma:.2f} / {W}  ln p(D|α,β)={log_ev:.2f}')
    alpha, beta = alpha_new, beta_new
print(f'\nconverged: α={alpha:.4f}, β={beta:.3f}, γ≈{log[-1]["gamma"]:.2f} of {W} parameters')
theta_final = theta_cur

## 8. Inspect the Hessian spectrum

γ partitions eigenvalues into "data-determined" (λ ≫ α, contribute ~1) and "prior-determined" (λ ≪ α, contribute ~0).

In [ ]:
H_final = hessian(sse, theta_final).detach().numpy()
H_final = 0.5 * (H_final + H_final.T)
eigs_bH = np.sort(np.clip(np.linalg.eigvalsh(beta * H_final), 0, None))[::-1]
frac = eigs_bH / (alpha + eigs_bH)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
axes[0].bar(np.arange(len(eigs_bH)), eigs_bH, color='#1B2D3D')
axes[0].axhline(alpha, color='#B44', ls='--', label=f'α = {alpha:.2f}')
axes[0].set_yscale('log'); axes[0].set_xlabel('eigenvalue index'); axes[0].set_ylabel('λ (of βH)')
axes[0].set_title('Data curvature vs prior'); axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].bar(np.arange(len(frac)), frac, color='#D4A24C', edgecolor='#1B2D3D')
axes[1].set_xlabel('eigenvalue index'); axes[1].set_ylabel('λ/(α+λ)')
axes[1].set_title(f'γ = Σ λ/(α+λ) ≈ {frac.sum():.1f} of {W}'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Final predictive band

With the converged α, β and Hessian, the predictive band tightens over the training support and fans out in the gap and at the tails.

In [ ]:
xs_wide = np.linspace(-1.5, 1.5, 500)
mean_w, std_w = predictive_band(theta_final, H_final, alpha, beta, xs_wide)

plt.figure(figsize=(8.5, 4.4))
plt.axvspan(x_train_np.min(), x_train_np.max(), color='#CCC', alpha=0.25, label='training support')
plt.plot(xs_wide, np.sin(2*np.pi*xs_wide), 'k--', alpha=0.5, label='true')
plt.scatter(x_train_np, y_train_np, color='#D4A24C', edgecolor='#1B2D3D', s=40, zorder=5, label='observations')
plt.plot(xs_wide, mean_w, color='#B44', lw=2.2, label='posterior mean')
plt.fill_between(xs_wide, mean_w - 2*std_w, mean_w + 2*std_w, color='#B44', alpha=0.2, label='±2σ')
plt.xlabel('x'); plt.ylabel('y'); plt.ylim(-4, 4); plt.legend()
plt.grid(alpha=0.3); plt.title('Predictive uncertainty widens outside the training support')
plt.show()

## 10. Exercises

1. **ReLU breaks Laplace.** Replace `F.gelu` with `F.relu` and recompute `H`. Inspect the eigenvalue spectrum; observe that many eigenvalues collapse to zero because ReLU's second derivative is zero almost everywhere. What does that do to γ, to the predictive band, and to the log evidence?
2. **Prior sensitivity.** Freeze α at different values (0.01, 0.1, 1, 10, 100) without running the evidence loop. How does the predictive band change? At what α does the network become essentially fixed at its prior mean?
3. **Model comparison.** Re-run the evidence loop with 4, 8, 12, 20, 32 hidden units. Plot the converged log evidence against the number of hidden units. You should see Occam's razor emerge: too small underfits (low evidence), too large wastes parameters (also low evidence), and there is a sweet spot.
4. **Classification.** Replace the Gaussian likelihood with Bernoulli and the mean-squared error with binary cross-entropy. Implement the probit predictive from Bishop 5.190: $p(t=1|x, D) = \sigma(\kappa(\sigma_a^2) b^\top w_{\text{MAP}})$ with $\kappa(s) = (1 + \pi s/8)^{-1/2}$. Reproduce Bishop Figure 5.23 on a synthetic two-moons dataset.
5. **Sample from the posterior.** Draw $K = 20$ samples $\mathbf{w}_k \sim \mathcal{N}(\mathbf{w}_{\text{MAP}}, \mathbf{A}^{-1})$ via Cholesky and plot the resulting network outputs. Compare the spread of sampled curves with the closed-form linearised band. Where do they agree?